In [31]:
import pandas as pd
import numpy as np

# Load the completed merged dataset
df = pd.read_csv("C:/Users/nived/New folder (2)/AI-Business-Operations-Management-Platform/datasets/raw/final_project_module.csv")
# Display basic information
print("Dataset Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
display(df.head())

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

print("\nDuplicate Rows:", df.duplicated().sum())

Dataset Shape: (8016, 32)

Columns:
['task_id', 'project_id', 'employee_id', 'task_title', 'task_description', 'task_priority', 'task_status', 'task_start_date', 'task_deadline', 'required_skill', 'estimated_hours', 'hours_logged', 'progress_percentage', 'allocation_score', 'recommended_employee', 'project_name', 'description', 'start_date', 'deadline', 'status', 'risk_level', 'employee_name', 'email', 'department', 'job_role', 'experience_years', 'hire_date', 'skills', 'availability_status', 'workload_percentage', 'active_tasks', 'performance_score']

First 5 rows:


,task_id,project_id,employee_id,task_title,task_description,task_priority,task_status,task_start_date,task_deadline,required_skill,...,email,department,job_role,experience_years,hire_date,skills,availability_status,workload_percentage,active_tasks,performance_score
0,T001,P00041,EMP008,UI Design,UI Design for the Inventory Management System ...,Medium,In Progress,2026-11-13,27-11-2026,React,...,employee008@company.local,Engineering,Project Manager,14.3,2013-08-04,Django,Busy,89.7,4,74.8
1,T002,P00009,EMP048,Login Module,Login Module for the Customer Support Chatbot ...,Medium,Completed,2026-07-12,19-11-2026,JavaScript,...,employee048@company.local,Engineering,Senior Developer,7.7,2017-06-20,Computer Vision,Busy,36.0,1,51.9
2,T003,P00006,EMP038,Deployment,Deployment for the Social Media Analytics proj...,Medium,Not Started,2025-02-16,29-08-2026,Docker,...,employee038@company.local,IT,Developer,10.5,2024-03-19,"Machine Learning, Excel, SQL",Busy,81.4,8,74.6
3,T004,P00015,EMP033,Integration Testing,Integration Testing for the Cybersecurity Upgr...,Medium,In Progress,2025-11-11,28-08-2026,Testing,...,employee033@company.local,Operations,Analyst,5.7,2025-06-03,"Deep Learning, Data Analysis",Unknown,66.5,4,60.2
4,T005,P00027,EMP015,Security Testing,Security Testing for the Supply Chain Analytic...,Medium,In Progress,2026-10-24,08-11-2026,Testing,...,employee015@company.local,HR,Business Analyst,2.8,2015-01-25,"Finance, React, Django",Available,28.1,1,69.0



Data Types:
task_id                  object
project_id               object
employee_id              object
task_title               object
task_description         object
task_priority            object
task_status              object
task_start_date          object
task_deadline            object
required_skill           object
estimated_hours         float64
hours_logged            float64
progress_percentage     float64
allocation_score        float64
recommended_employee      int64
project_name             object
description              object
start_date               object
deadline                 object
status                   object
risk_level               object
employee_name            object
email                    object
department               object
job_role                 object
experience_years        float64
hire_date                object
skills                   object
availability_status      object
workload_percentage     float64
active_tasks              i

In [32]:
print("Project Risk Distribution:")
print(df["risk_level"].value_counts())

print("\nProject Status Distribution:")
print(df["status"].value_counts())

print("\nTask Status Distribution:")
print(df["task_status"].value_counts())

print("\nTask Priority Distribution:")
print(df["task_priority"].value_counts())

Project Risk Distribution:
risk_level
High        2018
Critical    2009
Medium      2008
Low         1981
Name: count, dtype: int64

Project Status Distribution:
status
Completed      1966
In Progress    1807
On Hold        1722
Delayed        1465
Planning       1056
Name: count, dtype: int64

Task Status Distribution:
task_status
In Progress    3627
Completed      2217
On Hold        1196
Not Started     641
Cancelled       335
Name: count, dtype: int64

Task Priority Distribution:
task_priority
High        2981
Medium      2568
Critical    1502
Low          965
Name: count, dtype: int64


In [33]:
risk_consistency = (
    df.groupby("project_id")["risk_level"]
      .nunique()
)

print("Projects with consistent risk labels:",
      (risk_consistency == 1).sum())

print("Projects with conflicting risk labels:",
      (risk_consistency > 1).sum())

Projects with consistent risk labels: 800
Projects with conflicting risk labels: 0


In [34]:
# ============================================================
# Create Project-Level Dataset
# ============================================================

# Aggregate task and employee information for each project
project_ml = df.groupby("project_id").agg(
    total_tasks=("task_id", "count"),
    avg_progress=("progress_percentage", "mean"),
    avg_estimated_hours=("estimated_hours", "mean"),
    avg_hours_logged=("hours_logged", "mean"),
    avg_allocation_score=("allocation_score", "mean"),
    avg_workload=("workload_percentage", "mean"),
    avg_experience=("experience_years", "mean"),
    avg_performance=("performance_score", "mean")
).reset_index()

# Add project information
project_info = df[
    ["project_id", "project_name", "status", "risk_level"]
].drop_duplicates("project_id")

project_ml = project_ml.merge(
    project_info,
    on="project_id",
    how="left"
)

# Display results
print("Project-Level Dataset Shape:", project_ml.shape)

print("\nColumns:")
print(project_ml.columns.tolist())

print("\nFirst 5 Projects:")
print(project_ml.head())

print("\nRisk Distribution:")
print(project_ml["risk_level"].value_counts())

print("\nStatus Distribution:")
print(project_ml["status"].value_counts())

Project-Level Dataset Shape: (800, 12)

Columns:
['project_id', 'total_tasks', 'avg_progress', 'avg_estimated_hours', 'avg_hours_logged', 'avg_allocation_score', 'avg_workload', 'avg_experience', 'avg_performance', 'project_name', 'status', 'risk_level']

First 5 Projects:
  project_id  total_tasks  avg_progress  avg_estimated_hours  \
0     P00001            9     38.333333            25.537778   
1     P00002            5     53.800000            26.926000   
2     P00003            5     74.400000            31.998000   
3     P00004           10     59.400000            22.349000   
4     P00005           16     66.250000            31.314375   

   avg_hours_logged  avg_allocation_score  avg_workload  avg_experience  \
0         12.772222             55.590000     52.822222        7.166667   
1         24.510000             52.712000     50.220000        4.080000   
2         24.772000             58.982000     52.280000        4.040000   
3         19.861000             59.461000

In [35]:
# ============================================================
# Create Risk Prediction Dataset
# ============================================================

risk_features = [
    "total_tasks",
    "avg_progress",
    "avg_estimated_hours",
    "avg_hours_logged",
    "avg_allocation_score",
    "avg_workload",
    "avg_experience",
    "avg_performance"
]

risk_dataset = project_ml[
    risk_features + ["risk_level"]
].copy()

print("Risk Dataset Shape:", risk_dataset.shape)

print("\nRisk Dataset Columns:")
print(risk_dataset.columns.tolist())

print("\nRisk Distribution:")
print(
    risk_dataset["risk_level"]
    .value_counts()
    .reindex(["Low", "Medium", "High", "Critical"])
)

print("\nMissing Values:")
print(risk_dataset.isnull().sum().sum())

print("\nDuplicate Rows:")
print(risk_dataset.duplicated().sum())

Risk Dataset Shape: (800, 9)

Risk Dataset Columns:
['total_tasks', 'avg_progress', 'avg_estimated_hours', 'avg_hours_logged', 'avg_allocation_score', 'avg_workload', 'avg_experience', 'avg_performance', 'risk_level']

Risk Distribution:
risk_level
Low         200
Medium      200
High        200
Critical    200
Name: count, dtype: int64

Missing Values:
0

Duplicate Rows:
0


In [38]:
# ============================================================
# Train XGBoost, LightGBM and CatBoost Risk Models
# ============================================================

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Encode risk levels for XGBoost and LightGBM
risk_mapping = {
    "Low": 0,
    "Medium": 1,
    "High": 2,
    "Critical": 3
}

y_train_encoded = y_train.map(risk_mapping)
y_test_encoded = y_test.map(risk_mapping)

# ------------------------------------------------------------
# XGBoost
# ------------------------------------------------------------

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class=4,
    eval_metric="mlogloss",
    random_state=42
)

xgb_model.fit(
    X_train,
    y_train_encoded
)

print("XGBoost model trained successfully.")


# ------------------------------------------------------------
# LightGBM
# ------------------------------------------------------------

lgbm_model = LGBMClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    num_leaves=15,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multiclass",
    random_state=42,
    verbosity=-1
)

lgbm_model.fit(
    X_train,
    y_train_encoded
)

print("LightGBM model trained successfully.")


# ------------------------------------------------------------
# CatBoost
# ------------------------------------------------------------

catboost_model = CatBoostClassifier(
    iterations=200,
    depth=4,
    learning_rate=0.05,
    loss_function="MultiClass",
    verbose=False,
    random_seed=42
)

catboost_model.fit(
    X_train,
    y_train
)

print("CatBoost model trained successfully.")

XGBoost model trained successfully.
LightGBM model trained successfully.
CatBoost model trained successfully.


In [37]:
%pip install xgboost lightgbm catboost

   ---------------------------------------- 0.0/48.9 MB ? eta -:--:--
   ---------------------------------------- 0.5/48.9 MB 6.6 MB/s eta 0:00:08
   - -------------------------------------- 1.6/48.9 MB 6.1 MB/s eta 0:00:08
   -- ------------------------------------- 2.9/48.9 MB 5.8 MB/s eta 0:00:08
   --- ------------------------------------ 3.9/48.9 MB 5.8 MB/s eta 0:00:08
   ---- ----------------------------------- 5.2/48.9 MB 5.7 MB/s eta 0:00:08
   ----- ---------------------------------- 6.3/48.9 MB 5.7 MB/s eta 0:00:08
   ----- ---------------------------------- 7.3/48.9 MB 5.7 MB/s eta 0:00:08
   ------- -------------------------------- 8.7/48.9 MB 5.7 MB/s eta 0:00:08
   ------- -------------------------------- 9.7/48.9 MB 5.6 MB/s eta 0:00:07
   -------- ------------------------------- 11.0/48.9 MB 5.6 MB/s eta 0:00:07
   --------- ------------------------------ 12.1/48.9 MB 5.6 MB/s eta 0:00:07
   ---------- ----------------------------- 13.1/48.9 MB 5.6 MB/s eta 0:00:07
   

In [39]:
# ============================================================
# Train-Test Split for Risk Prediction
# ============================================================

from sklearn.model_selection import train_test_split

# Features
X = risk_dataset[risk_features]

# Target
y = risk_dataset["risk_level"]

# Split the dataset
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

print("\nTraining Risk Distribution:")
print(
    y_train.value_counts()
    .reindex(["Low", "Medium", "High", "Critical"])
)

print("\nTesting Risk Distribution:")
print(
    y_test.value_counts()
    .reindex(["Low", "Medium", "High", "Critical"])
)

Training Data Shape: (640, 8)
Testing Data Shape: (160, 8)

Training Risk Distribution:
risk_level
Low         160
Medium      160
High        160
Critical    160
Name: count, dtype: int64

Testing Risk Distribution:
risk_level
Low         40
Medium      40
High        40
Critical    40
Name: count, dtype: int64


In [40]:
# ============================================================
# Train Random Forest Risk Model
# ============================================================

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=6,
    random_state=42
)

rf_model.fit(X_train, y_train)

print("Random Forest model trained successfully.")

Random Forest model trained successfully.


In [41]:
# ============================================================
# Train XGBoost, LightGBM and CatBoost Risk Models
# ============================================================

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

# Encode risk levels for XGBoost and LightGBM
risk_mapping = {
    "Low": 0,
    "Medium": 1,
    "High": 2,
    "Critical": 3
}

y_train_encoded = y_train.map(risk_mapping)

# ------------------------------------------------------------
# XGBoost
# ------------------------------------------------------------

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multi:softmax",
    num_class=4,
    eval_metric="mlogloss",
    random_state=42
)

xgb_model.fit(
    X_train,
    y_train_encoded
)

print("XGBoost model trained successfully.")


# ------------------------------------------------------------
# LightGBM
# ------------------------------------------------------------

lgbm_model = LGBMClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.05,
    num_leaves=15,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="multiclass",
    random_state=42,
    verbosity=-1
)

lgbm_model.fit(
    X_train,
    y_train_encoded
)

print("LightGBM model trained successfully.")


# ------------------------------------------------------------
# CatBoost
# ------------------------------------------------------------

catboost_model = CatBoostClassifier(
    iterations=200,
    depth=4,
    learning_rate=0.05,
    loss_function="MultiClass",
    verbose=False,
    random_seed=42
)

catboost_model.fit(
    X_train,
    y_train
)

print("CatBoost model trained successfully.")

XGBoost model trained successfully.
LightGBM model trained successfully.
CatBoost model trained successfully.


In [42]:
# ============================================================
# Evaluate All Four Risk Prediction Models
# ============================================================

from sklearn.metrics import accuracy_score, f1_score

# Random Forest predictions
rf_pred = rf_model.predict(X_test)

# XGBoost predictions
xgb_pred_encoded = xgb_model.predict(X_test)

xgb_pred = pd.Series(
    xgb_pred_encoded
).map({
    0: "Low",
    1: "Medium",
    2: "High",
    3: "Critical"
}).values

# LightGBM predictions
lgbm_pred_encoded = lgbm_model.predict(X_test)

lgbm_pred_encoded = np.asarray(
    lgbm_pred_encoded
).reshape(-1)

lgbm_pred = pd.Series(
    lgbm_pred_encoded
).map({
    0: "Low",
    1: "Medium",
    2: "High",
    3: "Critical"
}).values

# CatBoost predictions
cat_pred = catboost_model.predict(X_test)

cat_pred = np.asarray(
    cat_pred
).reshape(-1)

# ------------------------------------------------------------
# Create comparison table
# ------------------------------------------------------------

results = pd.DataFrame({
    "Model": [
        "Random Forest",
        "XGBoost",
        "LightGBM",
        "CatBoost"
    ],
    "Accuracy": [
        accuracy_score(y_test, rf_pred),
        accuracy_score(y_test, xgb_pred),
        accuracy_score(y_test, lgbm_pred),
        accuracy_score(y_test, cat_pred)
    ],
    "Macro_F1": [
        f1_score(y_test, rf_pred, average="macro"),
        f1_score(y_test, xgb_pred, average="macro"),
        f1_score(y_test, lgbm_pred, average="macro"),
        f1_score(y_test, cat_pred, average="macro")
    ]
})

# Sort by Macro F1
results = results.sort_values(
    by="Macro_F1",
    ascending=False
).reset_index(drop=True)

print("Risk Model Comparison:")
print(results.round(4))

Risk Model Comparison:
           Model  Accuracy  Macro_F1
0        XGBoost    0.9750    0.9751
1       CatBoost    0.9750    0.9751
2  Random Forest    0.9688    0.9689
3       LightGBM    0.9562    0.9562


In [43]:
# ============================================================
# Detailed Evaluation: XGBoost vs CatBoost
# ============================================================

from sklearn.metrics import classification_report, confusion_matrix

# ------------------------------------------------------------
# XGBoost
# ------------------------------------------------------------

print("=" * 60)
print("XGBOOST CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_test,
        xgb_pred,
        labels=["Low", "Medium", "High", "Critical"],
        zero_division=0
    )
)

print("XGBoost Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        xgb_pred,
        labels=["Low", "Medium", "High", "Critical"]
    )
)


# ------------------------------------------------------------
# CatBoost
# ------------------------------------------------------------

print("\n" + "=" * 60)
print("CATBOOST CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_test,
        cat_pred,
        labels=["Low", "Medium", "High", "Critical"],
        zero_division=0
    )
)

print("CatBoost Confusion Matrix:")
print(
    confusion_matrix(
        y_test,
        cat_pred,
        labels=["Low", "Medium", "High", "Critical"]
    )
)

XGBOOST CLASSIFICATION REPORT
              precision    recall  f1-score   support

         Low       0.97      0.97      0.97        40
      Medium       0.93      1.00      0.96        40
        High       1.00      0.95      0.97        40
    Critical       1.00      0.97      0.99        40

    accuracy                           0.97       160
   macro avg       0.98      0.97      0.98       160
weighted avg       0.98      0.97      0.98       160

XGBoost Confusion Matrix:
[[39  1  0  0]
 [ 0 40  0  0]
 [ 1  1 38  0]
 [ 0  1  0 39]]

CATBOOST CLASSIFICATION REPORT
              precision    recall  f1-score   support

         Low       0.97      0.97      0.97        40
      Medium       0.93      1.00      0.96        40
        High       1.00      0.95      0.97        40
    Critical       1.00      0.97      0.99        40

    accuracy                           0.97       160
   macro avg       0.98      0.97      0.98       160
weighted avg       0.98      0.97   

In [44]:
import joblib

joblib.dump(xgb_model, "risk_prediction_xgboost.pkl")

print("XGBoost Risk Prediction model saved successfully.")

XGBoost Risk Prediction model saved successfully.


In [45]:
def predict_risk(
    total_tasks,
    avg_progress,
    avg_estimated_hours,
    avg_hours_logged,
    avg_allocation_score,
    avg_workload,
    avg_experience,
    avg_performance
):
    
    input_data = [[
        total_tasks,
        avg_progress,
        avg_estimated_hours,
        avg_hours_logged,
        avg_allocation_score,
        avg_workload,
        avg_experience,
        avg_performance
    ]]
    
    prediction = xgb_model.predict(input_data)[0]
    
    risk_mapping_reverse = {
        0: "Low",
        1: "Medium",
        2: "High",
        3: "Critical"
    }
    
    return risk_mapping_reverse[prediction]

In [46]:
sample_project = risk_dataset.iloc[0]

predicted_risk = predict_risk(
    sample_project["total_tasks"],
    sample_project["avg_progress"],
    sample_project["avg_estimated_hours"],
    sample_project["avg_hours_logged"],
    sample_project["avg_allocation_score"],
    sample_project["avg_workload"],
    sample_project["avg_experience"],
    sample_project["avg_performance"]
)

print("Predicted Risk:", predicted_risk)
print("Actual Risk:", sample_project["risk_level"])

Predicted Risk: Medium
Actual Risk: Medium


In [47]:
sample_project = risk_dataset[risk_dataset["risk_level"] == "Critical"].iloc[0]

predicted_risk = predict_risk(
    sample_project["total_tasks"],
    sample_project["avg_progress"],
    sample_project["avg_estimated_hours"],
    sample_project["avg_hours_logged"],
    sample_project["avg_allocation_score"],
    sample_project["avg_workload"],
    sample_project["avg_experience"],
    sample_project["avg_performance"]
)

print("Predicted Risk:", predicted_risk)
print("Actual Risk:", sample_project["risk_level"])

Predicted Risk: Critical
Actual Risk: Critical
